# VentureScout · Notebook 1 : construction du jeu de données

**Objectif du projet.** Aider un investisseur en capital-risque à trier les startups qui viennent de faire une première petite levée de fonds (tour *angel* / *seed*). Pour chaque startup, on veut estimer la probabilité qu'elle lève ensuite auprès de **fonds de capital-risque (VC)** dans les **24 mois**. C'est une **classification binaire** sur données tabulaires (variables numériques et catégorielles).

**Ce notebook** transforme les tables brutes de Crunchbase (une ligne par levée, une ligne par investissement…) en **un seul tableau : une ligne par startup**, avec :
- des variables explicatives calculées **uniquement avec l'information connue à la date de la première levée** (pour éviter toute fuite d'information du futur) ;
- la variable cible `y` (1 = a levé auprès de VC dans les 24 mois, 0 sinon).

Le résultat est enregistré dans `data/processed/venturescout_dataset.csv`, qui sert de point de départ au notebook 2 (exploration).

**Source des données.** Export officiel de Crunchbase d'octobre 2013 (licence CC-BY à l'époque de l'export), 4 fichiers CSV : `companies`, `rounds`, `investments`, `acquisitions`.
- Copie d'origine : https://github.com/datahoarder/crunchbase-october-2013 (mise à disposition par Dan Nguyen)
- Copie versionnée dans notre dépôt : `data/raw/` (chargée directement depuis GitHub, rien à installer).

**Solutions existantes.** Plusieurs travaux prédisent le « succès » des startups sur Crunchbase (ex. Kaggle *Startup Success Prediction*), mais ils utilisent souvent comme cible le statut final (racheté / fermé), observé bien après les variables, ce qui crée des fuites d'information. Notre formulation (fenêtre fixe de 24 mois après une date T, variables figées à T) évite ce problème.

In [1]:
# Imports (versions testées : pandas 2.x, numpy 1.26+/2.x)
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)

## 1. Chargement des données brutes

Les fichiers sont lus depuis le dossier local `data/` s'il existe (exécution dans le dépôt), sinon directement depuis GitHub (exécution dans Google Colab).

In [2]:
BASE_URL = "https://raw.githubusercontent.com/samiguessoum/hedgehogs-credit-risk-ml/main/venturescout/data/"

def lire_csv(chemin, **kwargs):
    local = Path("../data") / chemin
    source = local if local.exists() else BASE_URL + chemin
    return pd.read_csv(source, encoding="latin-1", low_memory=False, **kwargs)

companies = lire_csv("raw/crunchbase-companies.csv")
rounds = lire_csv("raw/crunchbase-rounds.csv")
invest = lire_csv("raw/crunchbase-investments.csv")
acq = lire_csv("raw/crunchbase-acquisitions.csv")

for nom, df in [("companies", companies), ("rounds", rounds), ("investments", invest), ("acquisitions", acq)]:
    print(f"{nom:13s} {df.shape[0]:>6d} lignes x {df.shape[1]:>2d} colonnes")

companies      17727 lignes x 17 colonnes
rounds         31679 lignes x 13 colonnes
investments    52870 lignes x 20 colonnes
acquisitions    6185 lignes x 20 colonnes


In [3]:
# Conversion des dates
rounds["date"] = pd.to_datetime(rounds["funded_at"], errors="coerce")
invest["date"] = pd.to_datetime(invest["funded_at"], errors="coerce")
acq["date"] = pd.to_datetime(acq["acquired_at"], errors="coerce")
companies["founded"] = pd.to_datetime(companies["founded_at"], errors="coerce")

# Type d'investisseur, lu dans le préfixe de son identifiant :
# /financial-organization/... = fonds, /person/... = business angel, /company/... = entreprise
invest["investor_type"] = invest["investor_permalink"].str.split("/").str[1]

DATE_FIN_DONNEES = rounds["date"].max()
print("Dernière date présente dans les données :", DATE_FIN_DONNEES.date())
print(rounds["funding_round_type"].value_counts())

Dernière date présente dans les données : 2013-10-01
funding_round_type
venture           7941
angel             6644
series-a          6523
series-b          3460
series-c+         3361
other             2921
private-equity     757
post-ipo            42
crowdfunding        30
Name: count, dtype: int64


## 2. Définir la date d'observation T et la cible y

**Population.** On garde les startups dont **la toute première levée est un tour `angel`** (dans cet export, les tours *seed* sont classés en `angel`). La date de cette levée est **T**.

**Fenêtre d'observation.** Les données s'arrêtent le 1er octobre 2013. Pour observer 24 mois complets après T, il faut **T ≤ 1er octobre 2011**. On commence en 2005 car avant, Crunchbase est très peu renseigné.

**Cible.** `y = 1` si la startup réalise, dans les 24 mois qui suivent T, un tour de type `series-a`, `series-b`, `series-c+` ou `venture` : ce sont, par définition, les tours menés par des fonds de capital-risque. Sinon `y = 0`.

On vérifie aussi une définition alternative fondée sur les investisseurs (« au moins un fonds parmi les investisseurs d'un tour ultérieur ») : elle est moins fiable car la liste des investisseurs manque pour une partie des tours (voir plus bas).

**Cas ambigus.** Une startup **rachetée** dans la fenêtre sans avoir levé auprès de VC n'a pas eu « l'occasion » de le faire : on l'**exclut** plutôt que de la compter comme un échec.

In [4]:
HORIZON = pd.DateOffset(months=24)
TYPES_VC = ["series-a", "series-b", "series-c+", "venture"]
T_MIN = pd.Timestamp("2005-01-01")
T_MAX = DATE_FIN_DONNEES - HORIZON

# Première levée de chaque startup, et date de son premier tour angel
premiere_levee = rounds.groupby("company_permalink")["date"].min()
premier_angel = rounds[rounds["funding_round_type"] == "angel"].groupby("company_permalink")["date"].min()

# Types de tours réalisés à la date de la première levée (pour exclure celles qui ont aussi un tour VC ce jour-là)
r0 = rounds.merge(premiere_levee.rename("T"), left_on="company_permalink", right_index=True)
types_a_T = r0[r0["date"] == r0["T"]].groupby("company_permalink")["funding_round_type"].agg(set)

pop = pd.DataFrame({"T": premier_angel})
pop = pop[pop["T"] == premiere_levee.reindex(pop.index)]                     # le 1er tour est un angel
pop = pop[types_a_T.reindex(pop.index).apply(lambda s: s <= {"angel", "other"})]  # pas de tour VC le même jour
print("Startups dont la 1re levée est un tour angel :", len(pop))
pop = pop[(pop["T"] >= T_MIN) & (pop["T"] <= T_MAX)]
print(f"... avec T entre {T_MIN.date()} et {T_MAX.date()} :", len(pop))

Startups dont la 1re levée est un tour angel : 5055
... avec T entre 2005-01-01 et 2011-10-01 : 2601


In [5]:
# Cible y : un tour VC dans ]T, T + 24 mois]
rv = rounds[rounds["funding_round_type"].isin(TYPES_VC)].merge(pop, left_on="company_permalink", right_index=True)
dans_fenetre = (rv["date"] > rv["T"]) & (rv["date"] <= rv["T"] + HORIZON)
pop["y"] = pop.index.isin(rv.loc[dans_fenetre, "company_permalink"]).astype(int)

# Définition alternative : un tour non-angel avec au moins un fonds parmi les investisseurs
iv = invest[(invest["investor_type"] == "financial-organization") & (invest["funding_round_type"] != "angel")]
iv = iv.merge(pop[["T"]], left_on="company_permalink", right_index=True)
ok = (iv["date"] > iv["T"]) & (iv["date"] <= iv["T"] + HORIZON)
pop["y_alt_investisseurs"] = pop.index.isin(iv.loc[ok, "company_permalink"]).astype(int)

print("Taux de y = 1 (définition principale)     :", round(pop["y"].mean(), 3))
print("Taux de y = 1 (définition investisseurs)  :", round(pop["y_alt_investisseurs"].mean(), 3))
print("Accord entre les deux définitions         :", round((pop["y"] == pop["y_alt_investisseurs"]).mean(), 3))

Taux de y = 1 (définition principale)     : 0.243
Taux de y = 1 (définition investisseurs)  : 0.194
Accord entre les deux définitions         : 0.944


In [6]:
# Part des tours pour lesquels on connaît les investisseurs, par type de tour
cle = ["company_permalink", "funding_round_type", "funded_at"]
avec_inv = rounds.merge(invest[cle].drop_duplicates(), on=cle, how="left", indicator=True)
(avec_inv.assign(investisseurs_connus=avec_inv["_merge"] == "both")
         .groupby("funding_round_type")["investisseurs_connus"].mean().round(2).sort_values())

funding_round_type
crowdfunding      0.10
private-equity    0.22
other             0.23
post-ipo          0.29
venture           0.49
angel             0.53
series-a          0.77
series-c+         0.86
series-b          0.88
Name: investisseurs_connus, dtype: float64

Les investisseurs ne sont connus que pour ~77 % des tours `series-a` et ~49 % des tours `venture`. La définition fondée sur les investisseurs manquerait donc beaucoup de vrais « oui » : on garde la **définition par type de tour** comme cible principale.

In [7]:
# Exclusion des startups rachetées pendant la fenêtre (ou avant T) sans tour VC
premier_rachat = acq.groupby("company_permalink")["date"].min()
rachat = premier_rachat.reindex(pop.index)
ambigu = rachat.notna() & (rachat <= pop["T"] + HORIZON) & (pop["y"] == 0)
print("Startups rachetées sans tour VC dans la fenêtre (exclues) :", int(ambigu.sum()))
pop = pop[~ambigu].copy()
print("Population finale :", len(pop), "startups | taux de y = 1 :", round(pop["y"].mean(), 3))

Startups rachetées sans tour VC dans la fenêtre (exclues) : 98
Population finale : 2503 startups | taux de y = 1 : 0.252


## 3. Variables explicatives (toutes calculées à la date T)

| Famille | Variables |
|---|---|
| Profil | `categorie`, `etat`, `region`, `annee_T`, `age_a_T_mois`, `date_creation_connue` |
| Première levée | `montant_T_usd`, `nb_tours_T` |
| Investisseurs du tour T | `nb_investisseurs_T`, `nb_fonds_T`, `nb_business_angels_T`, `nb_entreprises_T`, `investisseurs_connus_T` |
| Qualité des investisseurs | `exp_max_investisseurs`, `exp_moy_investisseurs` (nombre de startups financées avant T), `succes_max_investisseurs` (nombre de leurs startups passées déjà parvenues à un tour VC avant T) |
| Dynamique du marché | `tours_secteur_12m`, `tours_vc_secteur_12m`, `tours_etat_12m` (activité de financement dans le même secteur / État sur les 12 mois avant T) |

**Règle d'or :** aucune variable n'utilise d'information postérieure à T (pas de statut final, pas de montant total levé, pas de date du dernier financement…).

In [8]:
info = companies.set_index("permalink")
df = pop.copy()
df["name"] = info["name"].reindex(df.index)
df["categorie"] = info["category_code"].reindex(df.index)
df["etat"] = info["state_code"].reindex(df.index)
df["region"] = info["region"].reindex(df.index)
df["annee_T"] = df["T"].dt.year
fondee = info["founded"].reindex(df.index)
df["age_a_T_mois"] = ((df["T"] - fondee).dt.days / 30.44).round(1)
df["date_creation_connue"] = fondee.notna().astype(int)

# Première levée : montant et nombre de tours enregistrés à la date T
rt = rounds.merge(df[["T"]], left_on="company_permalink", right_index=True)
rt = rt[rt["date"] == rt["T"]]
g = rt.groupby("company_permalink")
df["montant_T_usd"] = g["raised_amount_usd"].sum(min_count=1)
df["nb_tours_T"] = g.size()
df[["categorie", "etat", "annee_T", "age_a_T_mois", "montant_T_usd", "nb_tours_T"]].head()

,categorie,etat,annee_T,age_a_T_mois,montant_T_usd,nb_tours_T
company_permalink,,,,,,
/company/1,mobile,CA,2011,-0.4,1000050.0,1
/company/1000memories,web,CA,2010,-5.9,15000.0,1
/company/10bestthings,web,OH,2009,0.0,50000.0,1
/company/140fire,advertising,CA,2010,4.1,500000.0,1
/company/169-st,games_video,FL,2009,0.6,50000.0,1


In [9]:
# Investisseurs du tour T
it = invest.merge(df[["T"]], left_on="company_permalink", right_index=True)
it = it[it["date"] == it["T"]]
comptes = it.groupby(["company_permalink", "investor_type"])["investor_permalink"].nunique().unstack(fill_value=0)
df["nb_fonds_T"] = comptes.get("financial-organization", 0).reindex(df.index).fillna(0).astype(int)
df["nb_business_angels_T"] = comptes.get("person", 0).reindex(df.index).fillna(0).astype(int)
df["nb_entreprises_T"] = comptes.get("company", 0).reindex(df.index).fillna(0).astype(int)
df["nb_investisseurs_T"] = df[["nb_fonds_T", "nb_business_angels_T", "nb_entreprises_T"]].sum(axis=1)
df["investisseurs_connus_T"] = (df["nb_investisseurs_T"] > 0).astype(int)
print("Part des startups dont on connaît les investisseurs du tour T :", round(df["investisseurs_connus_T"].mean(), 3))

Part des startups dont on connaît les investisseurs du tour T : 0.574


In [10]:
# Qualité des investisseurs du tour T : expérience et succès passés, mesurés AVANT T
# 1er investissement de chaque investisseur dans chaque startup
paires = invest.groupby(["investor_permalink", "company_permalink"])["date"].min().reset_index(name="date_inv")
# 1er tour VC de chaque startup (toutes startups confondues)
premier_vc = rounds[rounds["funding_round_type"].isin(TYPES_VC)].groupby("company_permalink")["date"].min()
paires["date_vc"] = paires["company_permalink"].map(premier_vc)

inv_T = it[["company_permalink", "investor_permalink", "T"]].drop_duplicates()
x = inv_T.merge(paires.rename(columns={"company_permalink": "autre_startup"}), on="investor_permalink")
x = x[x["date_inv"] < x["T"]]                                   # investissements passés uniquement
x["succes"] = (x["date_vc"] < x["T"]).astype(int)               # startup passée déjà arrivée au tour VC avant T
par_inv = x.groupby(["company_permalink", "investor_permalink"]).agg(exp=("autre_startup", "nunique"), succes=("succes", "sum"))
par_inv = par_inv.reindex(pd.MultiIndex.from_frame(inv_T[["company_permalink", "investor_permalink"]]), fill_value=0)
agg = par_inv.groupby("company_permalink").agg(exp_max=("exp", "max"), exp_moy=("exp", "mean"), succes_max=("succes", "max"))

df["exp_max_investisseurs"] = agg["exp_max"].reindex(df.index)
df["exp_moy_investisseurs"] = agg["exp_moy"].reindex(df.index).round(2)
df["succes_max_investisseurs"] = agg["succes_max"].reindex(df.index)
df[["nb_investisseurs_T", "exp_max_investisseurs", "succes_max_investisseurs"]].describe().round(2)

,nb_investisseurs_T,exp_max_investisseurs,succes_max_investisseurs
count,2503.00,1436.00,1436.00
mean,1.28,22.77,14.69
std,2.23,39.93,30.84
min,0.00,0.00,0.00
25%,0.00,0.00,0.00
50%,1.00,4.00,2.00
75%,1.00,26.00,14.00
max,26.00,240.00,182.00


In [11]:
# Dynamique du marché : nombre de tours dans le même secteur / État pendant les 12 mois avant T
def activite_12m(df, rounds, col_df, col_rounds, filtre=None):
    rr = rounds if filtre is None else rounds[filtre]
    dates = {k: np.sort(v.values) for k, v in rr.groupby(col_rounds)["date"]}
    res = []
    for cle, t in zip(df[col_df], df["T"]):
        d = dates.get(cle)
        if d is None:
            res.append(0)
            continue
        res.append(np.searchsorted(d, t.to_datetime64(), "left") - np.searchsorted(d, (t - pd.DateOffset(months=12)).to_datetime64(), "left"))
    return res

df["tours_secteur_12m"] = activite_12m(df, rounds, "categorie", "company_category_code")
df["tours_vc_secteur_12m"] = activite_12m(df, rounds, "categorie", "company_category_code", rounds["funding_round_type"].isin(TYPES_VC))
df["tours_etat_12m"] = activite_12m(df, rounds, "etat", "company_state_code")
df[["tours_secteur_12m", "tours_vc_secteur_12m", "tours_etat_12m"]].describe().round(1)

,tours_secteur_12m,tours_vc_secteur_12m,tours_etat_12m
count,2503.0,2503.0,2503.0
mean,178.2,124.0,577.8
std,132.8,96.5,574.3
min,0.0,0.0,0.0
25%,61.0,41.0,53.0
50%,151.0,113.0,351.0
75%,288.5,167.0,1139.0
max,548.0,407.0,1595.0


## 4. Enregistrement du tableau final

On ne fait **ici aucun nettoyage** (pas d'imputation, pas de suppression de valeurs aberrantes) : ces choix sont étudiés dans le notebook 2, puis appliqués dans les pipelines de modélisation **sur le jeu d'entraînement uniquement**.

In [12]:
colonnes = ["name", "T", "annee_T", "categorie", "etat", "region", "age_a_T_mois", "date_creation_connue",
            "montant_T_usd", "nb_tours_T", "nb_investisseurs_T", "nb_fonds_T", "nb_business_angels_T",
            "nb_entreprises_T", "investisseurs_connus_T", "exp_max_investisseurs", "exp_moy_investisseurs",
            "succes_max_investisseurs", "tours_secteur_12m", "tours_vc_secteur_12m", "tours_etat_12m",
            "y_alt_investisseurs", "y"]
final = df[colonnes].rename_axis("permalink").reset_index()
final["T"] = final["T"].dt.date

sortie = Path("../data/processed/venturescout_dataset.csv")
if sortie.parent.exists():
    final.to_csv(sortie, index=False)
    print("Enregistré :", sortie)
print(final.shape)
final.head()

Enregistré : ../data/processed/venturescout_dataset.csv
(2503, 24)


,permalink,name,T,annee_T,categorie,etat,region,age_a_T_mois,date_creation_connue,montant_T_usd,nb_tours_T,nb_investisseurs_T,nb_fonds_T,nb_business_angels_T,nb_entreprises_T,investisseurs_connus_T,exp_max_investisseurs,exp_moy_investisseurs,succes_max_investisseurs,tours_secteur_12m,tours_vc_secteur_12m,tours_etat_12m,y_alt_investisseurs,y
0,/company/1,One Inc.,2011-07-20,2011,mobile,CA,SF Bay,-0.4,1,1000050.0,1,8,4,4,0,1,222.0,70.25,168.0,231,128,1505,0,0
1,/company/1000memories,1000memories,2010-01-01,2010,web,CA,SF Bay,-5.9,1,15000.0,1,2,1,0,1,1,77.0,48.00,16.0,257,144,1204,1,1
2,/company/10bestthings,10BestThings,2009-04-01,2009,web,OH,Cleveland,0.0,1,50000.0,1,0,0,0,0,0,NaN,NaN,NaN,272,169,22,0,0
3,/company/140fire,140Fire,2010-05-05,2010,advertising,CA,Los Angeles,4.1,1,500000.0,1,0,0,0,0,0,NaN,NaN,NaN,203,145,1471,0,0
4,/company/169-st,169 ST.,2009-06-01,2009,games_video,FL,Orlando,0.6,1,50000.0,1,0,0,0,0,0,NaN,NaN,NaN,109,79,36,0,0
